## Đọc file log coi có hình k tải đc k

In [3]:
import pandas as pd
import re

def parse_log_file(file_path):
    """
    Đọc và phân tích file log tại đường dẫn được cung cấp.
    """
    # Pattern chung để bắt các thành phần cơ bản của log line
    log_pattern = re.compile(
        r'^(?P<Timestamp>\d{4}-\d{2}-\d{2}\s\d{2}:\d{2}:\d{2},\d{3})\s-\s(?P<Level>[A-Z]+)\s-\s\[(?P<ID>\d+)\]\s(?P<Message>.*)$'
    )

    data = []
    
    # *** ĐIỂM SỬA ĐỔI QUAN TRỌNG: Thay thế vòng lặp 'for line in log_content.strip().split('\n')' ***
    try:
        # Mở file log và đọc từng dòng
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                match = log_pattern.match(line.strip())
                if match:
                    record = match.groupdict()
                    message = record['Message'].strip()
                    record['Status'] = 'N/A'
                    record['URL'] = 'N/A'

                    # Phân tích dòng báo lỗi cố gắng (Attempt)
                    if 'Attempt' in message and 'failed:' in message:
                        parts = message.split('failed:', 1)
                        if len(parts) == 2:
                            record['Attempt_Info'] = parts[0].strip()
                            url_error = parts[1].strip()
                            
                            if ' - ' in url_error and 'for url:' in url_error:
                                url_and_status = url_error.split(' for url:')[0].strip()
                                url = url_and_status.split(' - ')[0].strip()
                                # Lấy phần status sau dấu gạch ngang đầu tiên
                                status = url_and_status.split(' - ', 1)[1].strip() 
                                record['URL'] = url
                                record['Status'] = status
                            record['Message'] = message

                    # Phân tích dòng báo lỗi vĩnh viễn (**Failed permanently**)
                    elif '**Failed permanently**' in message:
                        record['Status'] = 'Failed permanently'
                        # Lấy URL sau dấu hai chấm đầu tiên
                        url_part = message.split(':', 1)[1].strip()
                        record['URL'] = url_part
                        record['Attempt_Info'] = 'Attempt 3/3' 

                    data.append(record)
    except FileNotFoundError:
        print(f"Lỗi: Không tìm thấy file tại đường dẫn: {file_path}")
        return pd.DataFrame() # Trả về DataFrame trống nếu file không tồn tại
    except Exception as e:
        print(f"Lỗi khi đọc file log: {e}")
        return pd.DataFrame()
    # ******************************************************************************

    # Tạo DataFrame
    df = pd.DataFrame(data)
    if df.empty:
        return df
        
    # Chuyển cột Timestamp sang định dạng datetime
    df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='%Y-%m-%d %H:%M:%S,%f')
    return df

def filter_dataframe(df):
    """
    Áp dụng các điều kiện lọc theo yêu cầu.
    """
    if df.empty:
        print("DataFrame trống, không có gì để lọc.")
        return df
        
    print("--- DataFrame ban đầu (5 dòng đầu) ---")
    print(df[['Timestamp', 'URL', 'Status', 'Attempt_Info']].head())

    print("\n" + "="*50 + "\n")

    # 1. Điều kiện loại bỏ 404 (KHÔNG chứa '404 Client Error')
    # ~ là toán tử NOT trong pandas
    not_404_error = ~df['Status'].str.contains('404 Client Error', na=False)

    # 2. Điều kiện loại bỏ failed 3 lần (KHÔNG chứa '3/3' và KHÔNG chứa 'Failed permanently')
    not_attempt_3_of_3 = ~df['Attempt_Info'].str.contains('3/3', na=False)
    not_failed_permanently = ~df['Status'].str.contains('Failed permanently', na=False)

    # Kết hợp cả 3 điều kiện (sử dụng toán tử & (AND))
    df_filtered = df[not_404_error & not_attempt_3_of_3 & not_failed_permanently]

    print("--- DataFrame đã được lọc theo yêu cầu ---")
    # Hiển thị các cột quan trọng
    print(df_filtered[['Timestamp', 'URL', 'Status', 'Attempt_Info']])

    if df_filtered.empty:
        print("\nLƯU Ý: DataFrame kết quả trống.")
        
    return df_filtered

# --- THAY THẾ CHẠY CODE ---
# **Tên file log của bạn cần đọc:**
FILE_LOG_PATH = 'Baby_Products.jsonl_download_errors.log' 

# 1. Đọc và phân tích file log
log_df = parse_log_file(FILE_LOG_PATH)

# 2. Áp dụng Điều kiện Lọc
if not log_df.empty:
    df_result = filter_dataframe(log_df)

--- DataFrame ban đầu (5 dòng đầu) ---
                Timestamp                                                URL  \
0 2025-12-04 10:18:36.078  https://m.media-amazon.com/images/I/51UNp1kzAy...   
1 2025-12-04 10:18:36.498  https://m.media-amazon.com/images/I/51jVLzcc9A...   
2 2025-12-04 10:18:36.552  https://m.media-amazon.com/images/I/6115+441F6...   
3 2025-12-04 10:18:36.656  https://m.media-amazon.com/images/I/61ky6pn8AS...   
4 2025-12-04 10:18:36.697  https://m.media-amazon.com/images/I/714ygT6x8b...   

                        Status Attempt_Info  
0  404 Client Error: Not Found  Attempt 1/3  
1  404 Client Error: Not Found  Attempt 1/3  
2  404 Client Error: Not Found  Attempt 1/3  
3  404 Client Error: Not Found  Attempt 1/3  
4  404 Client Error: Not Found  Attempt 1/3  


--- DataFrame đã được lọc theo yêu cầu ---
                    Timestamp  URL Status Attempt_Info
1416  2025-12-04 10:38:36.716  N/A    N/A  Attempt 1/3
1417  2025-12-04 10:38:36.716  N/A    N/A  Attemp

In [8]:
df_result[df_result["Attempt_Info"].str.contains("2/3")]

,Timestamp,Level,ID,Message,Status,URL,Attempt_Info
1883,2025-12-04 11:20:03.812,ERROR,391496,Attempt 2/3 failed: https://images-na.ssl-imag...,N/A,N/A,Attempt 2/3
1884,2025-12-04 11:20:03.840,ERROR,391496,Attempt 2/3 failed: https://images-na.ssl-imag...,N/A,N/A,Attempt 2/3
1885,2025-12-04 11:20:04.327,ERROR,391496,Attempt 2/3 failed: https://images-na.ssl-imag...,N/A,N/A,Attempt 2/3
1886,2025-12-04 11:20:04.411,ERROR,391496,Attempt 2/3 failed: https://images-na.ssl-imag...,N/A,N/A,Attempt 2/3
1887,2025-12-04 11:20:04.459,ERROR,391496,Attempt 2/3 failed: https://images-na.ssl-imag...,N/A,N/A,Attempt 2/3
...,...,...,...,...,...,...,...
6653,2025-12-04 16:27:46.678,ERROR,2470059,Attempt 2/3 failed: https://images-na.ssl-imag...,N/A,N/A,Attempt 2/3
6654,2025-12-04 16:27:46.887,ERROR,2470198,Attempt 2/3 failed: https://images-na.ssl-imag...,N/A,N/A,Attempt 2/3
6655,2025-12-04 16:27:46.899,ERROR,2470143,Attempt 2/3 failed: https://images-na.ssl-imag...,N/A,N/A,Attempt 2/3
6656,2025-12-04 16:27:46.968,ERROR,2470210,Attempt 2/3 failed: https://images-na.ssl-imag...,N/A,N/A,Attempt 2/3


# Dữ liệu sản phẩm amazon

In [4]:
# %pip install -p requirements.txt

In [1]:
import os
import pandas as pd

In [2]:
PATH = "."
FILE_NAME = "Baby_Products.jsonl"
FILE_PATH = os.path.join(PATH, "data", FILE_NAME)

## Dữ liệu bình luận 

In [3]:
chunks = pd.read_json(os.path.join(PATH, "data", "Baby_Products.jsonl"), lines=True, chunksize=100_000)

In [4]:
# for chunk in chunks:
#     print(type(chunk))
df = next(chunks)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 10 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   rating             100000 non-null  int64         
 1   title              100000 non-null  object        
 2   text               100000 non-null  object        
 3   images             100000 non-null  object        
 4   asin               100000 non-null  object        
 5   parent_asin        100000 non-null  object        
 6   user_id            100000 non-null  object        
 7   timestamp          100000 non-null  datetime64[ns]
 8   helpful_vote       100000 non-null  int64         
 9   verified_purchase  100000 non-null  bool          
dtypes: bool(1), datetime64[ns](1), int64(2), object(6)
memory usage: 7.0+ MB


In [6]:
df.describe()

,rating,timestamp,helpful_vote
count,100000.000000,100000,100000.000000
mean,4.349860,2018-05-18 20:03:35.601761792,1.139410
min,1.000000,2001-10-11 16:38:59,0.000000
25%,4.000000,2016-04-17 15:06:40.500000,0.000000
50%,5.000000,2018-08-03 11:04:18.013500160,0.000000
75%,5.000000,2020-08-01 02:22:54.510249984,0.000000
max,5.000000,2023-03-19 05:08:04.640000,1725.000000
std,1.146504,NaN,12.794151


In [7]:
df[df["images"].str.len() > 0].iloc[1]["images"]

[{'small_image_url': 'https://m.media-amazon.com/images/I/61qN8WAWkyL._SL256_.jpg',
  'medium_image_url': 'https://m.media-amazon.com/images/I/61qN8WAWkyL._SL800_.jpg',
  'large_image_url': 'https://m.media-amazon.com/images/I/61qN8WAWkyL._SL1600_.jpg',
  'attachment_type': 'IMAGE'}]

In [9]:
df.iloc[15]["images"]

[{'small_image_url': 'https://m.media-amazon.com/images/I/61qN8WAWkyL._SL256_.jpg',
  'medium_image_url': 'https://m.media-amazon.com/images/I/61qN8WAWkyL._SL800_.jpg',
  'large_image_url': 'https://m.media-amazon.com/images/I/61qN8WAWkyL._SL1600_.jpg',
  'attachment_type': 'IMAGE'}]

### Tải hình của reviews

In [2]:
import os
import json
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

# ---------------------
# Logging: chỉ ghi vào file, không ghi console
# ---------------------
import logging

def setup_logging(log_file="download_errors.log"):
    logger = logging.getLogger("ImageDownloader")
    logger.setLevel(logging.DEBUG)

    file_handler = logging.FileHandler(log_file, mode='a', encoding='utf-8')
    file_handler.setLevel(logging.ERROR)
    formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    file_handler.setFormatter(formatter)

    logger.addHandler(file_handler)
    logger.propagate = False
    return logger

logger = setup_logging(f"{FILE_NAME}_download_errors.log")


# ---------------------
# Hàm tải 1 ảnh
# ---------------------
HEADERS = {
    'User-Agent': 'Mozilla/5.0',
    'Accept': 'image/webp,image/*,*/*;q=0.8'
}

# def download_one_image(url, save_path, idx):
#     try:
#         resp = requests.get(url, headers=HEADERS, timeout=10)
#         resp.raise_for_status()
#         with open(save_path, "wb") as f:
#             f.write(resp.content)
#     except Exception as e:
#         logger.error(f"[{idx}] Failed: {url} - {e}")

import time
import random
import requests

def download_one_image(url, save_path, idx, max_retry=3):
    for attempt in range(1, max_retry + 1):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=10)
            resp.raise_for_status()

            with open(save_path, "wb") as f:
                f.write(resp.content)

            return  # success, thoát hàm

        except Exception as e:
            logger.error(f"[{idx}] Attempt {attempt}/{max_retry} failed: {url} - {e}")

            if attempt == max_retry:
                logger.error(f"[{idx}] **Failed permanently**: {url}")
                return

            # exponential backoff + jitter
            sleep_time = (2 ** (attempt - 1)) + random.uniform(0, 0.5)
            time.sleep(sleep_time)


# ---------------------
# Hàm xử lý 1 dòng trong DataFrame
# ---------------------
def process_row(idx, row, root="downloaded_images"):
    images = row["images"]

    # Nếu không có ảnh → bỏ qua
    if not isinstance(images, list) or len(images) == 0:
        return

    # Tạo thư mục theo index của DF
    folder = os.path.join(root, str(idx))
    os.makedirs(folder, exist_ok=True)

    tasks = []

    # Mỗi item trong list images
    for i, img in enumerate(images):

        # Chọn URL ưu tiên: large → medium → small
        url = (
            img.get("large_image_url") or
            img.get("medium_image_url") or
            img.get("small_image_url")
        )

        if not url:
            continue

        ext = os.path.splitext(url)[1]
        if not ext:
            ext = ".jpg"

        filename = os.path.join(folder, f"img_{i}{ext}")

        tasks.append((url, filename))

    return tasks  # danh sách ảnh cần tải

# ---------------------
# Xử lý toàn bộ chunk song song
# ---------------------
def process_chunk(df, root="downloaded_images", max_workers=20):
    download_jobs = []

    # Tạo danh sách tất cả ảnh cần tải
    for idx, row in df.iterrows():
        tasks = process_row(idx, row, root)
        if tasks:
            download_jobs.extend([(idx, url, path) for url, path in tasks])

    # Tải song song
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [
            executor.submit(download_one_image, url, path, idx)
            for idx, url, path in download_jobs
        ]

        for f in as_completed(futures):
            pass  # không in gì để nhanh


# ---------------------
# Hàm đọc JSONL theo chunk và chạy
# ---------------------
def process_jsonl_in_chunks(filepath, chunksize=50000, root="downloaded_images"):
    chunks = pd.read_json(filepath, lines=True, chunksize=chunksize)

    for chunk_id, df in enumerate(chunks):
        print(f"Processing chunk {chunk_id}...")
        process_chunk(df, root=root)

# ---------------------
# CHẠY
# ---------------------

process_jsonl_in_chunks(FILE_PATH, chunksize=200_000, root=os.path.join(PATH, "data", FILE_NAME.replace(".jsonl", "_images")))

Processing chunk 0...
Processing chunk 1...
Processing chunk 2...
Processing chunk 3...
Processing chunk 4...
Processing chunk 5...
Processing chunk 6...
Processing chunk 7...
Processing chunk 8...
Processing chunk 9...
Processing chunk 10...
Processing chunk 11...
Processing chunk 12...
Processing chunk 13...
Processing chunk 14...
Processing chunk 15...
Processing chunk 16...
Processing chunk 17...
Processing chunk 18...
Processing chunk 19...
Processing chunk 20...
Processing chunk 21...
Processing chunk 22...
Processing chunk 23...
Processing chunk 24...
Processing chunk 25...
Processing chunk 26...
Processing chunk 27...
Processing chunk 28...
Processing chunk 29...
Processing chunk 30...


In [3]:
import os
import tarfile

images_root = os.path.join(PATH, "data", "Baby_Products_images")
output_dir = os.path.join(PATH, "data", "compressed_chunks")

os.makedirs(output_dir, exist_ok=True)

chunk_folders = sorted(os.listdir(images_root), key=lambda x: int(x.split('_')[1]))
chunk_folders

['chunk_10',
 'chunk_11',
 'chunk_12',
 'chunk_13',
 'chunk_14',
 'chunk_15',
 'chunk_16',
 'chunk_17',
 'chunk_18',
 'chunk_19',
 'chunk_20',
 'chunk_21',
 'chunk_22',
 'chunk_23',
 'chunk_24',
 'chunk_25',
 'chunk_26',
 'chunk_27',
 'chunk_28',
 'chunk_29',
 'chunk_30']

In [9]:
for chunk_folder in chunk_folders[:]:
    chunk_path = os.path.join(images_root, chunk_folder)
    
    if os.path.isdir(chunk_path):
        output_file = os.path.join(output_dir, f"{chunk_folder}.tar.gz")
        
        # Create tar.gz archive
        with tarfile.open(output_file, "w:gz") as tar:
            tar.add(chunk_path, arcname=chunk_folder)
        
        print(f"Compressed: {output_file}")

print("Chunks compressed successfully!")

Compressed: .\data\compressed_chunks\chunk_10.tar.gz
Compressed: .\data\compressed_chunks\chunk_11.tar.gz
Compressed: .\data\compressed_chunks\chunk_12.tar.gz
Compressed: .\data\compressed_chunks\chunk_13.tar.gz
Compressed: .\data\compressed_chunks\chunk_14.tar.gz
Compressed: .\data\compressed_chunks\chunk_15.tar.gz
Compressed: .\data\compressed_chunks\chunk_16.tar.gz
Compressed: .\data\compressed_chunks\chunk_17.tar.gz
Compressed: .\data\compressed_chunks\chunk_18.tar.gz
Compressed: .\data\compressed_chunks\chunk_19.tar.gz
Compressed: .\data\compressed_chunks\chunk_20.tar.gz
Compressed: .\data\compressed_chunks\chunk_21.tar.gz
Compressed: .\data\compressed_chunks\chunk_22.tar.gz
Compressed: .\data\compressed_chunks\chunk_23.tar.gz
Compressed: .\data\compressed_chunks\chunk_24.tar.gz
Compressed: .\data\compressed_chunks\chunk_25.tar.gz
Compressed: .\data\compressed_chunks\chunk_26.tar.gz
Compressed: .\data\compressed_chunks\chunk_27.tar.gz
Compressed: .\data\compressed_chunks\chunk_28.

In [5]:
import os
import shutil
import math

# =========================
# CẤU HÌNH BẮT BUỘC
# =========================

# Kích thước chunk đã dùng khi đọc file JSONL
CHUNK_SIZE = 200_000 

# Thư mục gốc chứa TẤT CẢ các thư mục idx (ví dụ: Baby_Products_images/15, Baby_Products_images/16,...)
SOURCE_ROOT = "./data/Baby_Products_images" 

# Thư mục gốc chứa các chunk đích (ví dụ: download_chunks/chunk_0, download_chunks/chunk_1,...)
TARGET_ROOT = "./data/download_chunks" 


def reorganize_product_folders():
    """
    Di chuyển các thư mục sản phẩm (theo idx) vào thư mục chunk tương ứng.
    """
    print(f"Bắt đầu tái tổ chức thư mục từ '{SOURCE_ROOT}' sang '{TARGET_ROOT}'...")
    print(f"CHUNK_SIZE đã đặt: {CHUNK_SIZE}")
    
    if not os.path.exists(SOURCE_ROOT):
        print(f"LỖI: Thư mục nguồn '{SOURCE_ROOT}' không tồn tại. Vui lòng kiểm tra lại đường dẫn.")
        return

    # Đảm bảo thư mục đích tồn tại
    os.makedirs(TARGET_ROOT, exist_ok=True)
    
    moved_count = 0
    
    # Lặp qua tất cả các mục trong thư mục nguồn
    for item_name in os.listdir(SOURCE_ROOT):
        source_path = os.path.join(SOURCE_ROOT, item_name)

        # 1. Kiểm tra xem có phải là thư mục và có tên là số nguyên không
        if os.path.isdir(source_path) and item_name.isdigit():
            try:
                # Lấy chỉ mục toàn cục của sản phẩm
                product_idx = int(item_name) 
                
                # 2. TÍNH TOÁN CHUNK ID TƯƠNG ỨNG: chunk_idx = floor(product_idx / CHUNK_SIZE)
                chunk_idx = product_idx // CHUNK_SIZE 
                
                # 3. Định nghĩa đường dẫn đích
                chunk_folder_name = f"chunk_{chunk_idx}"
                target_chunk_dir = os.path.join(TARGET_ROOT, chunk_folder_name)
                
                # Tạo thư mục chunk đích nếu chưa có
                os.makedirs(target_chunk_dir, exist_ok=True)
                
                # Đường dẫn đích cuối cùng cho thư mục sản phẩm
                target_path = os.path.join(target_chunk_dir, item_name)

                # 4. THỰC HIỆN DI CHUYỂN
                if os.path.exists(target_path):
                    print(f"CẢNH BÁO: Thư mục đích đã tồn tại: {target_path}. Bỏ qua di chuyển thư mục {source_path}")
                    continue

                shutil.move(source_path, target_path)
                moved_count += 1
                
                # Chỉ in ra thông báo cơ bản
                # print(f"Di chuyển idx {product_idx} -> {chunk_folder_name}/{item_name}")

            except Exception as e:
                print(f"LỖI khi xử lý thư mục {item_name}: {e}")
                
        # Loại bỏ log debug về việc bỏ qua các file/folder không phải số
            
    print("--------------------------------------------------")
    print(f"🎉 HOÀN TẤT! Đã di chuyển thành công {moved_count} thư mục sản phẩm.")
    print(f"Các thư mục idx hiện nằm trong '{TARGET_ROOT}'.")

if __name__ == "__main__":
    reorganize_product_folders()

Bắt đầu tái tổ chức thư mục từ './data/Baby_Products_images' sang './data/download_chunks'...
CHUNK_SIZE đã đặt: 200000
--------------------------------------------------
🎉 HOÀN TẤT! Đã di chuyển thành công 367523 thư mục sản phẩm.
Các thư mục idx hiện nằm trong './data/download_chunks'.
